# Study the use of BERT for Topic Modeling
Based on "BERTopic: Neural topic modeling with a class-based TF-IDF procedure"
Reference: https://arxiv.org/pdf/2203.05794  

This notebook will do the following:
1. Train the BerTopic implemented in bertopic python library following on similar or same data as mentioned in section 5.1 of the paper. One model per dataset or domain so we wil have 4 models in following instance:

    - a) New20group (static)
    - b) BBC (static)
    Then do similar validation on temporal data
    - c) Trump tweets 2021
    - d) UN Debates

Sample processed cleaned data fromat for static files:

| Document         | Clean Text     | # word count |
|--------------|-----------|------------|
| broadband ahead join internet .... | broadband ahead join internet ...  |71      |


2. Train  function is :
```
   topic_model = BERTopic(verbose=True)
   topics, probs = topic_model.fit_transform(docs)
```

3. Then do quantitative analysis as mentioned in section 5.3.:
```
  from bertopic.evaluation import evaluate_topic_coherence, evaluate_topic_diversity

  coherence = evaluate_topic_coherence(topic_model.get_topics(), docs)
  diversity = evaluate_topic_diversity(topic_model.get_topics())

  print(f"Coherence: {coherence:.4f}")
  print(f"Diversity: {diversity:.4f}")
```
Notes bertopic.evaluation is still a beta version does not get installed by default , you have install from git directly:
```
!pip install git+https://github.com/MaartenGr/BERTopic.git
```
Use topic_model.get_topic_info() and compute it manually using Gensim:
```
  from gensim.models.coherencemodel import CoherenceModel
  from gensim.corpora import Dictionary

  # Get topics as word lists
  topics = topic_model.get_topics()
  topic_words = [[word for word, _ in topics[topic]] for topic in topics]

  # Prepare texts for coherence model (same ones used for BERTopic)
  texts = [doc.split() for doc in docs]  # assumes simple tokenization

  dictionary = Dictionary(texts)
  coherence_model = CoherenceModel(
      topics=topic_words,
      texts=texts,
      dictionary=dictionary,
      coherence='c_v'
  )
  coherence_score = coherence_model.get_coherence()
  print(f"Coherence Score: {coherence_score:.4f}")
```

4. Then do  qualitative analysis as mentioned in section 5.3.:
```
  topic_model.get_topic_info().head()
  topic_model.get_topic(0)  # Top words for topic 0
```

5. Do Visualization of topics using:
```
   topic_model.visualize_topics()
   topic_model.visualize_barchart(top_n_topics=10)
   topic_model.visualize_hierarchy()
```


6. Save Model:
```
   topic_model.save("<DomainName>_bertopic_model")
```

7. Model Inference on unseen data
Here we create some unseen daat manually and see the inference. `topic_model.transform(unseen_docs)` will do the inference and returns a h TopicID. Then `topic_model.get_topic(topics[i])` returns a tuple : top keyword that describes the topic and Relevance score.

  Releavance Score also called importance or weigh. BERTopic, these numbers come from a class-based TF-IDF variant called c-TF-IDF. It reflects:
  How representative or important the word is for that specific topic, compared to all other topics.
  Higher numbers = more defining for the topic.
```
  # Create a list of unseen documents manually
  unseen_docs = [
    "The government announced a new policy on education reform today.",
    "Manchester United won their match after a thrilling penalty shootout.",
    "Scientists discovered a new exoplanet that could support life.",
    "The singer released a new charity single to raise funds for disaster relief.",
    "A major tech company unveiled its latest AI-powered smartphone."

    topics, probs = topic_model.transform(unseen_docs)

    for i, doc in enumerate(unseen_docs):
      topic_id = topics[i]

      print(f"\nDocument {i+1}: {doc}")
      print(f"Assigned topic: {topic_id}")
      print("Top words:", topic_model.get_topic(topics[i]))
]

```


## 1. Setup and Dependencies

### Optional: Mount Google Drive
Uncomment and run if you want to access files from your Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Install dependencies
!pip install bertopic
!pip install umap-learn hdbscan
!pip install gensim
!pip uninstall -y numpy gensim
!pip install numpy --upgrade
!pip install gensim --upgrade

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: gensim 4.3.3
Uninstalling gensim-4.3.3:
  Successfully uninstalled gensim-4.3.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 84.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.2.5 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.5 which is incompatible.
  Using cached gensim-4.3.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (8.1 kB)
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached gensim-4.3.3-cp311-cp311-manylinux_2_17_x

In [3]:
processed_data_path = "/content/drive/MyDrive/DLFA_IIsc_AIML/capstone_project/data_processed/" #set data path here
model_path="/content/drive/MyDrive/DLFA_IIsc_AIML/capstone_project/model/"
#data_path = "/content/drive/MyDrive/DLFA_IIsc_AIML/capstone_project/data/"

## 2. Fit BERTopic (Training for a domain or dataset)

In [4]:
#load data BBC
import pandas as pd
bbc_file = processed_data_path + "bbc_news_mindLab_corpus.csv"
df = pd.read_csv(bbc_file)  # adjust the path
docs = df['clean_text'].dropna().tolist()  # adjust column name as needed

In [5]:
# Print first 5 cleaned documents
for i, doc in enumerate(docs[:5]):
    print(f"Document {i+1}:\n{doc}\n{'-'*40}")


Document 1:
broadband ahead join internet fast accord official figure number business connect jump report broadband connection end compare nation rank world telecom body election campaign ensure affordable high speed net access american accord report broadband increasingly popular research shopping download music watch video total number business broadband rise end compare hook broadband subscriber line technology ordinary phone line support high datum speed cable lead account line broadband phone line connection accord figure
----------------------------------------
Document 2:
plan share sale owner technology dominate index plan sell share public list market operate accord document file stock market plan raise sale observer step close public icon technology boom recently pour cold water suggestion company sell share private technically public stock start trade list equity trade money sale investor buy share private filing document share technology firm company high growth potential s

In [6]:
from bertopic import BERTopic

topic_model = BERTopic(verbose=True)

In [7]:
topics, probs = topic_model.fit_transform(docs) #return list of topic and list of topic probablities per doc

2025-05-03 09:04:43,165 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/70 [00:00<?, ?it/s]

2025-05-03 09:07:27,383 - BERTopic - Embedding - Completed ✓
2025-05-03 09:07:27,384 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-05-03 09:07:47,654 - BERTopic - Dimensionality - Completed ✓
2025-05-03 09:07:47,658 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-05-03 09:07:47,793 - BERTopic - Cluster - Completed ✓
2025-05-03 09:07:47,802 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-05-03 09:07:48,249 - BERTopic - Representation - Completed ✓


## 3. Quantitative Validation

Topic Coherence measures how semantically related the top words in each topic are. A higher score means:

Top words in a topic frequently appear together in the original documents.

The topic is more interpretable to humans.

In [14]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary

# Get topics as word lists
topics = topic_model.get_topics()
topic_words = [[word for word, _ in topics[topic]] for topic in topics]

# Prepare texts for coherence model (same ones used for BERTopic)
texts = [doc.split() for doc in docs]  # assumes simple tokenization

dictionary = Dictionary(texts)
coherence_model = CoherenceModel(
    topics=topic_words,
    texts=texts,
    dictionary=dictionary,
    coherence='c_v'
)
coherence_score = coherence_model.get_coherence()
print(f"Coherence Score: {coherence_score:.4f}")

Coherence Score: 0.7157


In [9]:
# from bertopic.evaluation import evaluate_topic_coherence, evaluate_topic_diversity

# coherence = evaluate_topic_coherence(topic_model.get_topics(), docs)
# diversity = evaluate_topic_diversity(topic_model.get_topics())

# print(f"Coherence: {coherence:.4f}")
# print(f"Diversity: {diversity:.4f}")

## 4. Qualitative Validation

In [15]:
topic_model.get_topic_info().head()
topic_model.get_topic(0)  # Top words for topic 0


[('election', 0.02581841032750082),
 ('party', 0.023605645277193044),
 ('government', 0.022771997725588463),
 ('labour', 0.02248835289721983),
 ('tory', 0.020187492941636586),
 ('plan', 0.016205751118657897),
 ('minister', 0.01496118689513081),
 ('leader', 0.013879773926794348),
 ('public', 0.013448897014796167),
 ('tax', 0.01307130831609563)]

In [16]:
#Representative Docs
topic_model.get_representative_docs(0)[:3]  # Inspect 3 sample docs from topic 0

['rally labour voter issue rally cry supporter warn stake high stay home protest vote general election chancellor poll expect fall clear fundamental choice labour investment tory cut party spring conference tory win conservative lib dem insist voter face high taxi test labour pack audience accuse plot cut equivalent sack teacher country lie conservative record government promise return mistake inflation interest rate lose reserve negative equity tory boom bust central divide line election conservative party plan deep cut service labour government forward platform stability reform renew hospital school public service proud spend turn economy chancellor promise continue economic stability growth term power pledge continue fight child pensioner poverty promise young property message young couple wait obtain home housing centre manifesto labour government match low mortgage rate buyer speech prompt promise end teenage unemployment highlight plan debt relief world poor country national mini

## 5. Visualization

In [24]:
from IPython.display import display

display(topic_model.visualize_topics())
display(topic_model.visualize_barchart(top_n_topics=10))
display(topic_model.visualize_hierarchy())

In [25]:
# Exclude topic -1 (outliers)
num_topics = len([topic for topic in topic_model.get_topic_info().Topic if topic != -1])
print(f"Total number of topics (excluding outliers): {num_topics}")

Total number of topics (excluding outliers): 42


## 6. Save Model

In [18]:
bbc_model_file = model_path + "bbc_news_model"
topic_model.save(bbc_model_file)

2025-05-03 09:11:39,651 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


## 7. Inference or model validation with unseen data
Later, You Can Use Inference
After fit_transform(), you can use transform(new_docs) on unseen data:

In [21]:
new_topics, new_probs = topic_model.transform(["Fans around the world are downloading the new charity single released to support global disaster relief. Featuring international stars and backed by major record labels, the song has already topped charts in several countries. Music stores are reporting high demand, and digital platforms have seen a surge in streaming. Proceeds will go toward humanitarian aid, with organizers hopeful the track will raise millions. Industry experts compare the success to the original Band Aid release, which became a cultural milestone decades ago."])
print(f"Assigned topic: {new_topics[0]}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-05-03 09:38:30,032 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2025-05-03 09:38:30,047 - BERTopic - Dimensionality - Completed ✓
2025-05-03 09:38:30,048 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2025-05-03 09:38:30,050 - BERTopic - Cluster - Completed ✓


Assigned topic: 2


The `topic_word` hold the tuple " top keyword that describes the topic and Relevance score.
#### Releavance Score also called importance or weigh. BERTopic, these numbers come from a class-based TF-IDF variant called c-TF-IDF. It reflects:

- How representative or important the word is for that specific topic,
compared to all other topics.

Higher numbers = more defining for the topic.

In [23]:
topic_words = topic_model.get_topic(new_topics[0])
print(f"Top words for topic {new_topics[0]}:  {topic_words}")

Top words for topic 2:  [('music', 0.05307212506268715), ('band', 0.051828047139879985), ('song', 0.046468661529554374), ('album', 0.03831537863020457), ('artist', 0.028872394432138824), ('singer', 0.02657478895322101), ('single', 0.026480588877024102), ('good', 0.02631497789860735), ('award', 0.02441501872875063), ('record', 0.024333394008757843)]


# Make a lookup where topic ID has a human readable Name
While BERTopic gives you numeric topic IDs (like 2, 5, 7), those numbers by themselves don't have meaning — they just group similar documents.

To make sense of these topics, you look at their top keywords, and based on that, you assign a human-readable name to each topic.

Python code that automatically generates a table of topic IDs, their top keywords, and a suggested name (based on top 3–5 keywords) from your trained BERTopic model.

In [33]:
# Number of top words per topic to use for naming
top_n_words = 5

# Get all topic IDs (excluding outlier -1)
topic_info = topic_model.get_topic_info()
valid_topics = topic_info[topic_info.Topic != -1].Topic.tolist()

# Build a table with topic ID, top words, and label
topic_labels = []

for topic_id in valid_topics:
    words = topic_model.get_topic(topic_id)
    top_words = [word for word, _ in words[:top_n_words]]
    label = ", ".join(top_words)
    topic_labels.append({
        "Topic ID": topic_id,
         "Topic Related To": f"{top_words[0].capitalize()}",
        "Top Words": label

    })

# Create and display a DataFrame
df_labels = pd.DataFrame(topic_labels)
print(df_labels.to_string(index=False))

 Topic ID Topic Related To                                     Top Words
        0         Election     election, party, government, labour, tory
        1             Film                film, award, actor, star, good
        2            Music              music, band, song, album, artist
        3             Seed                 seed, final, open, win, match
        4            Share             share, profit, firm, bid, company
        5          Economy         economy, rate, growth, rise, economic
        6             Game             game, console, gamer, title, play
        7             Race               race, indoor, win, olympic, run
        8            Fraud   fraud, firm, company, executive, accounting
        9             Club       club, manager, player, contract, season
       10           Injury              injury, play, nation, week, miss
       11          Referee        referee, match, player, game, incident
       12         Computer     computer, laptop, pr

In [34]:
# Create a dictionary from the DataFrame to map topic id to s
topic_name_map = dict(zip(df_labels["Topic ID"], df_labels["Topic Related To"]))

In [35]:
# Create a list of unseen documents manually
unseen_docs = [
    "The government announced a new policy on education reform today.",
    "Manchester United won their match after a thrilling penalty shootout.",
    "Scientists discovered a new exoplanet that could support life.",
    "The singer released a new charity single to raise funds for disaster relief.",
    "A major tech company unveiled its latest AI-powered smartphone."
]

In [36]:
topics, probs = topic_model.transform(unseen_docs)

for i, doc in enumerate(unseen_docs):
    topic_id = topics[i]
    topic_name = topic_name_map.get(topic_id, "Unknown or Outlier Topic")

    print(f"\nDocument {i+1}: {doc}")
    print(f"Assigned topic: {topic_id}")
    print(f"Topic Related To: {topic_name}")
    print("Top words:", topic_model.get_topic(topics[i]))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-05-03 10:10:39,534 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2025-05-03 10:10:39,568 - BERTopic - Dimensionality - Completed ✓
2025-05-03 10:10:39,569 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2025-05-03 10:10:39,571 - BERTopic - Cluster - Completed ✓



Document 1: The government announced a new policy on education reform today.
Assigned topic: 0
Topic Related To: Election
Top words: [('election', 0.02581841032750082), ('party', 0.023605645277193044), ('government', 0.022771997725588463), ('labour', 0.02248835289721983), ('tory', 0.020187492941636586), ('plan', 0.016205751118657897), ('minister', 0.01496118689513081), ('leader', 0.013879773926794348), ('public', 0.013448897014796167), ('tax', 0.01307130831609563)]

Document 2: Manchester United won their match after a thrilling penalty shootout.
Assigned topic: 14
Topic Related To: Arsenal
Top words: [('arsenal', 0.06610872364446378), ('league', 0.04815147338421253), ('goal', 0.04564050363459159), ('premiership', 0.04242230772602924), ('play', 0.04133362676068686), ('draw', 0.03649643140977832), ('win', 0.03641495969761637), ('team', 0.03446864668862373), ('season', 0.0341758940806081), ('tie', 0.0321273136807982)]

Document 3: Scientists discovered a new exoplanet that could support